In [16]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [17]:
# Load the merged dataset
df = pd.read_csv("../data/processed/diabetic_with_sdoh.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Readmission rate: {df['readmitted_30'].mean()*100:.1f}%")

Rows: 101,766
Columns: 48
Readmission rate: 11.2%


In [18]:
## Medication feature engineering

# Total number of diabetes medications being taken
diabetes_meds = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
                 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
                 'miglitol', 'troglitazone', 'tolazamide', 'insulin',
                 'glyburide-metformin', 'glipizide-metformin',
                 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
                 'metformin-pioglitazone']

# Count of active medications (non-zero values)
df['total_diabetes_meds'] = (df[diabetes_meds] > 0).sum(axis=1)

# Medication complexity score
df['med_complexity_score'] = (
    df['num_medications'] * 0.4 +
    df['total_diabetes_meds'] * 0.3 +
    df['num_procedures'] * 0.2 +
    df['number_diagnoses'] * 0.1
)

# High medication burden flag (top 25%)
med_threshold = df['num_medications'].quantile(0.75)
df['high_med_burden'] = (df['num_medications'] >= med_threshold).astype(int)

# Insulin flag
df['on_insulin'] = (df['insulin'] > 0).astype(int)

# Multiple drug changes flag
df['multiple_med_changes'] = (df['change'] > 0).astype(int)

In [19]:
print("Medication features engineered!")
print(f"New features added:")
print(f"  total_diabetes_meds   — mean: {df['total_diabetes_meds'].mean():.2f}")
print(f"  med_complexity_score  — mean: {df['med_complexity_score'].mean():.2f}")
print(f"  high_med_burden       — {df['high_med_burden'].sum():,} patients ({df['high_med_burden'].mean()*100:.1f}%)")
print(f"  on_insulin            — {df['on_insulin'].sum():,} patients ({df['on_insulin'].mean()*100:.1f}%)")
print(f"  multiple_med_changes  — {df['multiple_med_changes'].sum():,} patients ({df['multiple_med_changes'].mean()*100:.1f}%)")

Medication features engineered!
New features added:
  total_diabetes_meds   — mean: 12.86
  med_complexity_score  — mean: 11.28
  high_med_burden       — 27,571 patients (27.1%)
  on_insulin            — 89,548 patients (88.0%)
  multiple_med_changes  — 54,755 patients (53.8%)


## Engineer clinical risk features

In [20]:
## Engineer clinical risk features

# Additional clinical risk features
# Prior utilization score (how much they used healthcare before)
df['prior_utilization'] = (
    df['number_inpatient'] * 0.5 +
    df['number_emergency'] * 0.3 +
    df['number_outpatient'] * 0.2
)

# High risk patient flag
df['high_prior_inpatient'] = (df['number_inpatient'] >= 2).astype(int)

# Emergency admission flag
df['emergency_admission'] = (df['admission_source_id'] == 7).astype(int)

# Long stay flag (top 25% of hospital stays)
stay_threshold = df['time_in_hospital'].quantile(0.75)
df['long_stay'] = (df['time_in_hospital'] >= stay_threshold).astype(int)

# High diagnosis burden
df['high_diagnosis_burden'] = (df['number_diagnoses'] >= 7).astype(int)

print("Clinical risk features engineered!")
print(f"New features added:")
print(f"  prior_utilization      — mean: {df['prior_utilization'].mean():.2f}")
print(f"  high_prior_inpatient   — {df['high_prior_inpatient'].sum():,} patients ({df['high_prior_inpatient'].mean()*100:.1f}%)")
print(f"  emergency_admission    — {df['emergency_admission'].sum():,} patients ({df['emergency_admission'].mean()*100:.1f}%)")
print(f"  long_stay              — {df['long_stay'].sum():,} patients ({df['long_stay'].mean()*100:.1f}%)")
print(f"  high_diagnosis_burden  — {df['high_diagnosis_burden'].sum():,} patients ({df['high_diagnosis_burden'].mean()*100:.1f}%)")

Clinical risk features engineered!
New features added:
  prior_utilization      — mean: 0.45
  high_prior_inpatient   — 14,615 patients (14.4%)
  emergency_admission    — 57,494 patients (56.5%)
  long_stay              — 28,688 patients (28.2%)
  high_diagnosis_burden  — 70,598 patients (69.4%)


In [21]:
# Check final dataset shape and feature groups
print(f"Final dataset shape: {df.shape}")
print(f"\nFeature Groups ")

clinical_features = ['age', 'time_in_hospital', 'num_lab_procedures',
                     'num_procedures', 'num_medications', 'number_outpatient',
                     'number_emergency', 'number_inpatient', 'number_diagnoses',
                     'admission_type_id', 'discharge_disposition_id',
                     'admission_source_id', 'medical_specialty',
                     'diag_1', 'diag_2', 'diag_3']

sdoh_features = ['BPHIGH', 'CHECKUP', 'CHOLSCREEN', 
                 'DEPRESSION', 'DIABETES', 'OBESITY']

medication_features = ['total_diabetes_meds', 'med_complexity_score',
                      'high_med_burden', 'on_insulin', 
                      'multiple_med_changes']

risk_features = ['prior_utilization', 'high_prior_inpatient',
                 'emergency_admission', 'long_stay', 
                 'high_diagnosis_burden']

print(f"Clinical features: {len(clinical_features)}")
print(f"SDOH features: {len(sdoh_features)}")
print(f"Medication features: {len(medication_features)}")
print(f"Risk features: {len(risk_features)}")
print(f"Total features: {len(clinical_features) + len(sdoh_features) + len(medication_features) + len(risk_features)}")

Final dataset shape: (101766, 58)

Feature Groups 
Clinical features: 16
SDOH features: 6
Medication features: 5
Risk features: 5
Total features: 32


In [22]:
# Define all features for the model
all_features = clinical_features + sdoh_features + medication_features + risk_features

# Create final feature matrix
X = df[all_features]
y = df['readmitted_30']

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution before SMOTE:")
print(f"  Not Readmitted (0): {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)")
print(f"  Readmitted <30 (1): {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")

Feature matrix shape: (101766, 32)
Class distribution before SMOTE:
  Not Readmitted (0): 90,409 (88.8%)
  Readmitted <30 (1): 11,357 (11.2%)


In [23]:
# Apply SMOTE to balance classes
print("Applying SMOTE")

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"SMOTE applied!")
print(f"Class distribution after SMOTE:")
print(f"  Not Readmitted (0): {(y_resampled==0).sum():,} ({(y_resampled==0).mean()*100:.1f}%)")
print(f"  Readmitted <30 (1): {(y_resampled==1).sum():,} ({(y_resampled==1).mean()*100:.1f}%)")
print(f"Dataset size: {len(X_resampled):,} rows")

Applying SMOTE
SMOTE applied!
Class distribution after SMOTE:
  Not Readmitted (0): 90,409 (50.0%)
  Readmitted <30 (1): 90,409 (50.0%)
Dataset size: 180,818 rows


In [24]:
# Scale features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)

# Convert back to dataframe
X_scaled_df = pd.DataFrame(X_scaled, columns=all_features)

print("Features scaled!")
print(f"Shape: {X_scaled_df.shape}")
print(f"ample statistics after scaling:")
print(X_scaled_df[['age', 'time_in_hospital', 
                    'num_medications', 'OBESITY']].describe().round(2))

Features scaled!
Shape: (180818, 32)
ample statistics after scaling:
             age  time_in_hospital  num_medications    OBESITY
count  180818.00         180818.00        180818.00  180818.00
mean        0.00              0.00             0.00      -0.00
std         1.00              1.00             1.00       1.00
min        -3.88             -1.18            -1.92      -2.59
25%        -0.71             -0.82            -0.65      -0.60
50%        -0.08             -0.12            -0.15       0.27
75%         0.55              0.59             0.49       0.74
max         1.82              3.40             8.21       1.65


In [25]:
import joblib

# Save scaled features and target
X_scaled_df['readmitted_30'] = y_resampled.values
X_scaled_df.to_csv("../data/processed/feature_store.csv", index=False)

# Save unscaled resampled data too (needed for tree models)
X_unscaled_df = pd.DataFrame(X_resampled, columns=all_features)
X_unscaled_df['readmitted_30'] = y_resampled.values
X_unscaled_df.to_csv("../data/processed/feature_store_unscaled.csv", index=False)

# Save the scaler for later use in deployment
joblib.dump(scaler, "../models/scaler.pkl")

print("Feature store saved!")
print(f"Files saved:")
print(f"  data/processed/feature_store.csv — scaled (for LogReg)")
print(f"  data/processed/feature_store_unscaled.csv — unscaled (for XGBoost/RF)")
print(f"  models/scaler.pkl — scaler object")

Feature store saved!
Files saved:
  data/processed/feature_store.csv — scaled (for LogReg)
  data/processed/feature_store_unscaled.csv — unscaled (for XGBoost/RF)
  models/scaler.pkl — scaler object


In [26]:
import os

# Check all processed files
print("data/processed/")
for f in sorted(os.listdir("../data/processed/")):
    size = os.path.getsize(f"../data/processed/{f}")
    print(f"  {f} — {size/1024/1024:.1f} MB")

print("\nmodels/")
for f in sorted(os.listdir("../models/")):
    size = os.path.getsize(f"../models/{f}")
    print(f"  {f} — {size/1024:.1f} KB")

print("\ndashboard/")
for f in sorted(os.listdir("../dashboard/")):
    size = os.path.getsize(f"../dashboard/{f}")
    print(f"  {f} — {size/1024:.1f} KB")

data/processed/
  diabetic_cleaned.csv — 8.5 MB
  diabetic_with_sdoh.csv — 11.9 MB
  feature_store.csv — 108.4 MB
  feature_store_unscaled.csv — 24.2 MB
  sdoh_state_averages.csv — 0.0 MB

models/
  scaler.pkl — 2.1 KB

dashboard/
  age_analysis.png — 68.5 KB
  clinical_features.png — 149.4 KB
  correlation_heatmap.png — 180.7 KB
  diagnosis_analysis.png — 96.7 KB
  inpatient_analysis.png — 82.1 KB
  sdoh_features.png — 173.4 KB
  target_distribution.png — 73.2 KB
